# 02 — Dense PGD (Tier A) — Person 1, due Oct 20

Steps (docs/notebook_specs.md): (1) unit tests, (2) time one PGD step and size the fixed attack subset,
(3) pilot 30 probes then the full subset at ε ∈ {2, 4, 8, 16}/255, (4) `results/raw/pgd_results.csv`.

**Kaggle inputs:** Notebook 01's saved output (packed `CASIA-B-pkl`) · GPU T4 · Internet on ·
secret `GITHUB_TOKEN` (Contents read + write). Results are pushed to GitHub after every sweep point, so a
disconnect loses at most one probe; re-running resumes (same `run_id`).

**Done when:** unit tests pass on the real model; `results/processed/subset_attack.csv` written (shared by
every attack — never regenerated); subset size recorded in `configs/attacks.yaml`; all four ε done.

In [ ]:
# --- Paths per platform. Edit ONLY this cell. -------------------------------------------
import os, sys, subprocess
from pathlib import Path

PLATFORM = "kaggle" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else \
           "colab" if "google.colab" in sys.modules else "local"
PATHS = {
    # Kaggle: team repo cloned into /kaggle/working; data + checkpoints from the private team Dataset.
    "kaggle": dict(repo="/kaggle/working/cross_attack_gait",
                   opengait="/kaggle/working/OpenGait",
                   dataset_root="/kaggle/input/casia-b-opengait/CASIA-B-pkl",
                   ckpt_dir="/kaggle/input/casia-b-opengait/checkpoints"),
    # Colab: Drive mirror of the same Dataset.
    "colab": dict(repo="/content/cross_attack_gait",
                  opengait="/content/OpenGait",
                  dataset_root="/content/drive/MyDrive/gait/CASIA-B-pkl",
                  ckpt_dir="/content/drive/MyDrive/gait/checkpoints"),
    # Local: set CASIAB_PKL and GAIT_CKPT_DIR (e.g. ../data/CASIA-B-pkl). CPU-only machines cannot run
    # OpenGait models (BaseModel hardcodes CUDA); official gate runs on GPU.
    "local": dict(repo=str(Path.cwd().parent), opengait=str(Path.cwd().parent.parent / "OpenGait"),
                  dataset_root=os.environ.get("CASIAB_PKL", ""), ckpt_dir=os.environ.get("GAIT_CKPT_DIR", "")),
}[PLATFORM]
TEAM_REPO_URL = "https://github.com/haragam22/gait-paper.git"   # e.g. https://github.com/<org>/cross_attack_gait.git (private; use a token on Kaggle)

if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
if PLATFORM != "local" and not Path(PATHS["repo"]).exists():
    env = dict(os.environ)
    if PLATFORM == "kaggle":  # private repo: add a GitHub token as Kaggle secret GITHUB_TOKEN (Add-ons > Secrets)
        import base64
        from kaggle_secrets import UserSecretsClient
        auth = base64.b64encode(f"x-access-token:{UserSecretsClient().get_secret('GITHUB_TOKEN')}".encode()).decode()
        # token via env-only git config: never in argv, error messages, or .git/config (saved with notebook output)
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    subprocess.run(["git", "clone", TEAM_REPO_URL, PATHS["repo"]], check=True, env=env)
sys.path.insert(0, PATHS["repo"]); os.chdir(PATHS["repo"])

import src.utils as U
BASE = U.load_yaml("configs/baseline.yaml")
if not Path(PATHS["opengait"]).exists():
    subprocess.run(["git", "clone", "https://github.com/ShiqiYu/OpenGait.git", PATHS["opengait"]], check=True)
subprocess.run(["git", "-C", PATHS["opengait"], "checkout", "-q", BASE["opengait_commit"]], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "tensorboard"], check=True)

# Kaggle data: use a packed CASIA-B-pkl attached as input (team Dataset or this notebook's saved output);
# otherwise pack the public PNG mirror (trnquanghuyn/casia-b, attach as input) with OpenGait's own script,
# same command as docs/reproducibility.md (~20 min). "Save Version" keeps /kaggle/working/CASIA-B-pkl so
# teammates attach this notebook's output instead of re-packing.
if PLATFORM == "kaggle" and not Path(PATHS["dataset_root"]).exists():
    # Attached inputs mount at /kaggle/input/<slug>/ or deeper (datasets/<owner>/<slug>/, notebooks/...).
    find_input = lambda rel: (p for pat in ("*/", "*/*/", "*/*/*/", "*/*/*/*/") for p in Path("/kaggle/input").glob(pat + rel))
    attached = next(find_input("CASIA-B-pkl"), None)
    packed = Path("/kaggle/working/CASIA-B-pkl")
    if attached is None and not (packed / ".done").exists():
        raw = next(find_input("output/001"), None)
        if raw is None:
            tree = sorted(str(p) for pat in ("*", "*/*", "*/*/*") for p in Path("/kaggle/input").glob(pat) if p.is_dir())
            raise FileNotFoundError("No CASIA-B-pkl and no raw CASIA-B (output/001) under /kaggle/input. Attach "
                                    "Notebook 01's saved output (Add Input > Your Work) or trnquanghuyn/casia-b. "
                                    "Found:\n" + "\n".join(tree[:40]))
        raw = raw.parent
        subprocess.run([sys.executable, "datasets/pretreatment.py", "-i", str(raw), "-o", str(packed),
                        "-d", "GREW", "-n", str(os.cpu_count())], cwd=PATHS["opengait"], check=True)
        (packed / ".done").touch()
    PATHS["dataset_root"] = str(attached or packed)
print("dataset_root:", PATHS["dataset_root"])
BASE["opengait_dir"] = PATHS["opengait"]
BASE["dataset_root"] = PATHS["dataset_root"]

In [ ]:
# --- Cell 1 record: date, seed, versions, GPU, platform, OpenGait commit --------------
SEED = BASE["seed"]
U.seed_everything(SEED)
ENV = U.env_record(SEED, BASE["opengait_dir"], {"notebook": "02_pgd_attack"})
assert ENV["opengait_commit"] == BASE["opengait_commit"], ENV["opengait_commit"]
ATK = U.load_yaml("configs/attacks.yaml")
ENV

In [ ]:
# --- Checkpoint: from the team Dataset, else download the released one -----------------
import urllib.request
gb = BASE["models"]["gaitbase"]
ck = Path(PATHS["ckpt_dir"]) / "GaitBase_DA-60000.pt"
if not ck.exists():
    ck = Path("/kaggle/working" if PLATFORM == "kaggle" else ".") / "GaitBase_DA-60000.pt"
    if not ck.exists():
        urllib.request.urlretrieve(gb["checkpoint_url"], ck)
gb["checkpoint"] = str(ck)
print("GaitBase:", ck, U.file_sha256(ck)[:12])

In [ ]:
# --- Load GaitBase + clean gallery ---------------------------------------------------
import time, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from src.evaluation import load_model, Gallery, evaluate_attack, AttackTarget, read_results, summarize_runs
from src.data import stratified_subset, save_subset, load_subset
from src.attacks import masks as M
from src.attacks.pgd import PGD, dense, edge, temporal, margin_grad, run_tier_a
model, data, og_cfg = load_model(BASE, "gaitbase")
gallery = Gallery.build(model, data, cache=f"results/processed/gallery_{model.name}_{model.checkpoint_id}.npz")
ENV.update(checkpoint=gb["checkpoint"], checkpoint_id=model.checkpoint_id)
print(model.name, model.checkpoint_id, model.device, "gallery", tuple(gallery.emb.shape))

def push(msg):
    pushed = U.push_results(PATHS["repo"], msg, U.github_token(PLATFORM), paths=("results", "configs"))
    print("pushed" if pushed else "nothing to push")

## 1. Unit tests (CPU fake model) + the same checks on real GaitBase

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-600:]); assert r.returncode == 0, r.stdout[-3000:]

# Real model, 3 probes: eps=0 == clean; ||delta||_inf <= eps; range; weights unchanged (evaluate_attack asserts).
probe3 = stratified_subset(data.probes, 1, seed=SEED)
clean3 = evaluate_attack(model, data, gallery, probe3)
zero3 = evaluate_attack(model, data, gallery, probe3, PGD(0.0, steps=2), attack_name="pgd0")
assert (zero3.adv_margin.values == clean3.clean_margin.values).all() and (zero3.linf_actual == 0).all()
eps = 8 / 255
x, t = data.load(probe3[0]).to(model.device), AttackTarget.for_probe(gallery, probe3[0])
a = PGD(eps, steps=3, seed=SEED); xa, xb = a(model, x, t), a(model, x, t)
assert float((xa - x).abs().max()) <= eps + 1e-6 and 0 <= float(xa.min()) and float(xa.max()) <= 1
assert torch.equal(xa, xb), "not seed-stable"
print("real-model checks PASS")

## 2. Time one PGD step → size the fixed attack subset
One sweep = one (attack, ε). Tier A + Hou ≈ 34 sweeps (dense 4, edge 4, temporal 2 rules × 3 k × 4 ε, Hou 2).
Set `BUDGET_HOURS` to the GPU hours Person 1 can spend on all of them (Kaggle + Colab).
**Stop here and agree the size before running the next cell — the subset is fixed forever once written.**

In [ ]:
BUDGET_HOURS = 24      # total GPU-h for all Tier A + Hou sweeps; edit
N_SWEEPS = 34
timing = stratified_subset(data.probes, 2, seed=SEED + 1)        # 6 probes, 2 per condition
step_s, n_frames = [], []
for rec in timing:
    x, t = data.load(rec).to(model.device), AttackTarget.for_probe(gallery, rec)
    margin_grad(model, x, t)                                     # warm-up
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(5):
        margin_grad(model, x, t)
    torch.cuda.synchronize(); step_s.append((time.time() - t0) / 5); n_frames.append(x.shape[1])
sec_per_probe = ATK["steps"] * float(np.mean(step_s)) + 1.0     # + clean/adv embeds and I/O
N = U.size_subset(sec_per_probe, N_SWEEPS, BUDGET_HOURS)
print(f"one step {np.mean(step_s)*1000:.0f} ms (T={n_frames}); one probe ≈ {sec_per_probe:.1f} s")
print(f"→ {N} probes per condition ({3*N} total); one sweep ≈ {3*N*sec_per_probe/3600:.2f} h")
if N < 40:
    print("WARNING: < 40/condition leaves little room for the 20/condition fingerprint subset "
          "(probes attacked by ALL five attacks). Raise BUDGET_HOURS if possible.")

In [ ]:
# Fix the subset ONCE. If it already exists (pushed by an earlier run), reuse it — never regenerate.
SUBSET = Path("results/processed/subset_attack.csv")
if SUBSET.exists():
    subset = load_subset(SUBSET, data)
    print("existing subset:", len(subset), "probes")
else:
    subset = stratified_subset(data.probes, N, seed=SEED)
    save_subset(subset, SUBSET)
    U.set_yaml_scalar("configs/attacks.yaml", "attack_n_per_condition", N)
    print("wrote", SUBSET, len(subset), "probes; attacks.yaml attack_n_per_condition =", N)
    push("results: fixed attack subset (Notebook 02)")
pd.Series([r.condition for r in subset]).value_counts()

## 3. Pilot (10/condition, ε = 8/255), then the full sweep

In [ ]:
OUT = Path("results/raw/pgd_results.csv")
run = lambda e255, probes: run_tier_a(model, data, gallery, probes, "dense", dense(), e255, ATK, SEED, OUT,
                                      SUBSET, run_meta={"env": ENV})
n_pilot = ATK["subsets"]["pilot_n_per_condition"]
pilot = [r for c in ["NM", "BG", "CL"] for r in [s for s in subset if s.condition == c][:n_pilot]]
t0 = time.time(); p = run(8, pilot)       # same run_id as the full eps=8 run: these rows are reused
print(f"pilot {len(p)} probes in {(time.time()-t0)/60:.1f} min; success rate {p.attack_success.mean():.2f}")
p.groupby("condition")[["clean_rank1", "adv_rank1", "attack_success", "linf_actual"]].mean()

In [ ]:
# Full sweep. Resumable: re-run this cell (or the notebook) after a disconnect.
# Grid eps first, then the small extra eps for dense only (configs/attacks.yaml families.dense.extra_eps).
# Finished points are skipped (rows already in the CSV for that run_id).
for e255 in ATK["eps_grid"] + ATK["families"]["dense"]["extra_eps"]:
    t0 = time.time(); df = run(e255, subset)
    print(f"eps {e255}/255: {len(df)} probes, success {df.attack_success.mean():.3f}, {(time.time()-t0)/60:.1f} min")
    push(f"results: Notebook 02 dense PGD eps={e255}/255 ({ENV['platform']})")

## 4. Summary

In [ ]:
R = read_results("results/raw/pgd_results.csv")
R = R[R.run_id.str.endswith(model.checkpoint_id) & R.sample_id.isin([s.sample_id for s in subset])]
display(summarize_runs(R, ["eps_255", "condition"]).round(3))
fig, ax = plt.subplots(figsize=(5.5, 3.5))
R = R.assign(eps_255=(R.eps * 255).round(2))
for c, g in R.groupby("condition"):
    s = g.groupby("eps_255").adv_rank1_viewavg.mean() * 100
    ax.plot(s.index, s.values, marker="o", label=c)
ax.set_xlabel("ε × 255"); ax.set_ylabel("adversarial Rank-1 (%)"); ax.set_xscale("log", base=2)
ax.set_title("Dense PGD"); ax.legend(fontsize=7)
plt.tight_layout(); Path("results/figures").mkdir(exist_ok=True); plt.savefig("results/figures/02_pgd_dense_rank1.png", dpi=150)
push("results: Dense PGD summary figure")

## Findings, limitations, next step
Numbers from `results/raw/pgd_results.csv` (Kaggle T4, 2026-10-16/17).

- Unit tests: pass (CPU suite + real-model checks).
- Subset: 130 probes per condition (390 total), `results/processed/subset_attack.csv`, seed 0;
  `configs/attacks.yaml` attack_n_per_condition = 130. Step time: (fill from the timing cell output).
- Dense PGD **saturates at every ε in the grid**: attack success = 100% of clean-correct probes at
  ε = 2, 4, 8, 16 /255 (clean pooled Rank-1 = 95.9%); adversarial view-averaged Rank-1 ≤ 1.0% per condition
  at ε = 2/255 and 0.0% from ε = 8/255. ‖δ‖∞ = ε exactly; median margin −0.61 (clean) → +1.52 (ε = 2/255).
- Only 74–89% of pixels change: silhouette pixels sit at 0 or 1, so steps pushing outside [0, 1] are clipped.
- Limitation: no dynamic range in the grid → no matched-success-rate point for dense PGD vs weaker attacks.
- Next: Notebooks 03 / 04 on the same subset; decide on extra small-ε dense points (Notebook 08 needs them).